In [ ]:
import kagglehub

plantdisease_path = kagglehub.dataset_download("emmarex/plantdisease")

print("Path to plantdisease dataset files:", plantdisease_path)

Using Colab cache for faster access to the 'plantdisease' dataset.
Path to plantdisease dataset files: /kaggle/input/plantdisease


In [ ]:
import kagglehub

plantdoc_path = kagglehub.dataset_download("nirmalsankalana/plantdoc-dataset")

print("Path to plantdoc dataset files:", plantdoc_path)

100%|██████████| 896M/896M [00:06<00:00, 139MB/s]

Extracting files...


Path to plantdoc dataset files: /root/.cache/kagglehub/datasets/nirmalsankalana/plantdoc-dataset/versions/7


In [ ]:
import os

print(f"Contents of the plantdisease dataset directory ({plantdisease_path}):")
for item in os.listdir(plantdisease_path):
    print(f"- {item}")

print(f"\nContents of the plantdoc dataset directory ({plantdoc_path}):")
for item in os.listdir(plantdoc_path):
    print(f"- {item}")

Contents of the plantdisease dataset directory (/kaggle/input/plantdisease):
- PlantVillage
- plantvillage

Contents of the plantdoc dataset directory (/root/.cache/kagglehub/datasets/nirmalsankalana/plantdoc-dataset/versions/7):
- file_renamer.py
- train
- folder_renamer.py
- test


In [ ]:
import tensorflow as tf

image_size = (256, 256)
batch_size = 32

train_dir = os.path.join(plantdoc_path, 'train')

if os.path.exists(train_dir) and os.listdir(train_dir):
    print(f"Loading training data from: {train_dir}")
    train_ds = tf.keras.utils.image_dataset_from_directory(
        train_dir,
        labels='inferred',
        label_mode='int',
        image_size=image_size,
        interpolation='nearest',
        batch_size=batch_size,
        shuffle=True,
        seed=42
    )

    print(f"\nNumber of training batches: {len(train_ds)}")
    print(f"Number of classes: {len(train_ds.class_names)}")
    print(f"Class names: {train_ds.class_names}")
else:
    print(f"Training directory not found or is empty at: {train_dir}")
    print("Please ensure the 'train' subdirectory exists and contains image data.")


Loading training data from: /root/.cache/kagglehub/datasets/nirmalsankalana/plantdoc-dataset/versions/7/train
Found 2670 files belonging to 28 classes.

Number of training batches: 84
Number of classes: 28
Class names: ['Apple_Scab_Leaf', 'Apple_leaf', 'Apple_rust_leaf', 'Bell_pepper_leaf', 'Bell_pepper_leaf_spot', 'Blueberry_leaf', 'Cherry_leaf', 'Corn_Gray_leaf_spot', 'Corn_leaf_blight', 'Corn_rust_leaf', 'Peach_leaf', 'Potato_leaf_early_blight', 'Potato_leaf_late_blight', 'Raspberry_leaf', 'Soyabean_leaf', 'Squash_Powdery_mildew_leaf', 'Strawberry_leaf', 'Tomato_Early_blight_leaf', 'Tomato_Septoria_leaf_spot', 'Tomato_leaf', 'Tomato_leaf_bacterial_spot', 'Tomato_leaf_late_blight', 'Tomato_leaf_mosaic_virus', 'Tomato_leaf_yellow_virus', 'Tomato_mold_leaf', 'Tomato_two_spotted_spider_mites_leaf', 'grape_leaf', 'grape_leaf_black_rot']


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 4
plantvillage_data_path = os.path.join(plantdisease_path, 'PlantVillage')

try:
    train_village = tf.keras.utils.image_dataset_from_directory(
        plantvillage_data_path,
        validation_split=0.2,
        subset="training",
        seed=123,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE
    )

    val_village = tf.keras.utils.image_dataset_from_directory(
        plantvillage_data_path,
        validation_split=0.2,
        subset="validation",
        seed=123,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE
    )

    class_names = train_village.class_names
    num_classes = len(class_names)

    print(f"Number of classes found: {num_classes}")
    print(f"Class names: {class_names}")
except Exception as e:
    print(f"Error loading PlantVillage dataset: {e}")
    print(f"Please ensure the directory '{plantvillage_data_path}' exists and contains image data.")

Found 20638 files belonging to 15 classes.
Using 16511 files for training.
Found 20638 files belonging to 15 classes.
Using 4127 files for validation.
Number of classes found: 15
Class names: ['Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot', 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato__Target_Spot', 'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato__Tomato_mosaic_virus', 'Tomato_healthy']


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_village = train_village.shuffle(buffer_size=128).prefetch(buffer_size=AUTOTUNE)
val_village = val_village.prefetch(buffer_size=AUTOTUNE)

print("Datasets configured without caching to save RAM.")

Datasets configured without caching to save RAM.


### Optimizations for GPU and RAM

-   **Removed `.cache()`**: For large datasets, caching the entire dataset in RAM can lead to out-of-memory errors. By removing this, data will be loaded directly from disk as needed, reducing RAM consumption.
-   **Enabled Mixed Precision (`mixed_float16`)**: This policy uses `float16` (half-precision) wherever possible, which significantly reduces the memory footprint on the GPU and can also speed up computation on modern GPUs.

If you still encounter memory issues, you can try reducing the `BATCH_SIZE` (currently set to 32) in the cell where the datasets are loaded (`xOUrslyAXljI`).

In [ ]:
tf.keras.mixed_precision.set_global_policy('mixed_float16')
print("Mixed precision enabled.")

Mixed precision enabled.


In [ ]:
data_aug = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

In [ ]:
base_model = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)

base_model.trainable = False

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
inputs = keras.Input(shape=(224,224,3))

x = data_aug(inputs)
x = tf.keras.applications.efficientnet.preprocess_input(x)

x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = keras.Model(inputs, outputs)

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history1 = model.fit(
    train_village,
    validation_data=val_village,
    epochs=25
)

Epoch 1/25
4128/4128 ━━━━━━━━━━━━━━━━━━━━ 145s 28ms/step - accuracy: 0.8081 - loss: 0.6476 - val_accuracy: 0.8907 - val_loss: 0.3304
Epoch 2/25
4128/4128 ━━━━━━━━━━━━━━━━━━━━ 112s 25ms/step - accuracy: 0.8873 - loss: 0.3527 - val_accuracy: 0.8961 - val_loss: 0.2944
Epoch 3/25
4128/4128 ━━━━━━━━━━━━━━━━━━━━ 105s 25ms/step - accuracy: 0.8995 - loss: 0.3021 - val_accuracy: 0.9028 - val_loss: 0.2732
Epoch 4/25
4128/4128 ━━━━━━━━━━━━━━━━━━━━ 109s 26ms/step - accuracy: 0.9083 - loss: 0.2769 - val_accuracy: 0.9166 - val_loss: 0.2419
Epoch 5/25
4128/4128 ━━━━━━━━━━━━━━━━━━━━ 106s 25ms/step - accuracy: 0.9115 - loss: 0.2657 - val_accuracy: 0.9130 - val_loss: 0.2419
Epoch 6/25
4128/4128 ━━━━━━━━━━━━━━━━━━━━ 104s 25ms/step - accuracy: 0.9125 - loss: 0.2541 - val_accuracy: 0.9123 - val_loss: 0.2343
Epoch 7/25
4128/4128 ━━━━━━━━━━━━━━━━━━━━ 104s 25ms/step - accuracy: 0.9150 - loss: 0.2545 - val_accuracy: 0.9213 - val_loss: 0.2136
Epoch 8/25
4128/4128 ━━━━━━━━━━━━━━━━━━━━ 143s 25ms/step - accuracy: 

In [ ]:
model.save("plant_village_model.h5")

In [ ]:

train_doc = tf.keras.utils.image_dataset_from_directory(
    plantdoc_path,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_doc = tf.keras.utils.image_dataset_from_directory(
    plantdoc_path,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

train_doc = train_doc.cache().shuffle(1000).prefetch(AUTOTUNE)
val_doc = val_doc.cache().prefetch(AUTOTUNE)

Found 2922 files belonging to 2 classes.
Using 2338 files for training.
Found 2922 files belonging to 2 classes.
Using 584 files for validation.


In [ ]:
model = tf.keras.models.load_model("plant_village_model.h5")

In [ ]:
base_model = model.layers[2]  # EfficientNet inside model

base_model.trainable = True

# freeze early layers
for layer in base_model.layers[:-30]:
    layer.trainable = False

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history2 = model.fit(
    train_doc,
    validation_data=val_doc,
    epochs=5
)

Epoch 1/5
585/585 ━━━━━━━━━━━━━━━━━━━━ 92s 73ms/step - accuracy: 0.2742 - loss: 4.7950 - val_accuracy: 0.4041 - val_loss: 3.9438
Epoch 2/5
585/585 ━━━━━━━━━━━━━━━━━━━━ 19s 33ms/step - accuracy: 0.5997 - loss: 2.0242 - val_accuracy: 0.6062 - val_loss: 2.1864
Epoch 3/5
585/585 ━━━━━━━━━━━━━━━━━━━━ 18s 31ms/step - accuracy: 0.7472 - loss: 1.2911 - val_accuracy: 0.7003 - val_loss: 1.6152
Epoch 4/5
585/585 ━━━━━━━━━━━━━━━━━━━━ 20s 34ms/step - accuracy: 0.8199 - loss: 0.9362 - val_accuracy: 0.7329 - val_loss: 1.3274
Epoch 5/5
585/585 ━━━━━━━━━━━━━━━━━━━━ 19s 33ms/step - accuracy: 0.8362 - loss: 0.8522 - val_accuracy: 0.7637 - val_loss: 1.1203


In [ ]:
model.save("plant_disease_final_model.h5")

In [ ]:
from tensorflow.keras.preprocessing import image
import os

# Assuming plantdoc_path is defined from previous cells
# Re-defining image_size and class_names from the plantdoc dataset for accurate prediction
# This assumes the plantdoc_path was loaded as `train_ds` (or similar) earlier
# I will simulate getting a class_names list from the plantdoc_path's train directory

# Get class names from the plantdoc training directory
plantdoc_train_dir = os.path.join(plantdoc_path, 'train')
class_names = sorted(os.listdir(plantdoc_train_dir))

# Find an example image path from the plantdoc training directory
# We'll pick the first class and the first image in that class folder
if class_names and os.listdir(os.path.join(plantdoc_train_dir, class_names[0])):
    first_class_dir = os.path.join(plantdoc_train_dir, class_names[0])
    img_filename = os.listdir(first_class_dir)[0]
    img_path = os.path.join(first_class_dir, img_filename)
    print(f"Using image for prediction from: {img_path}")
else:
    print("Could not find an image in the plantdoc training directory for prediction.")
    # Fallback or error handling if no image is found
    img_path = ""

if img_path:
    img = image.load_img(img_path, target_size=(224,224))
    img = image.img_to_array(img)
    img = np.expand_dims(img, axis=0)

    pred = model.predict(img)
    print("Predicted class:", class_names[np.argmax(pred)])
else:
    print("Prediction skipped as no valid image path was found.")

Using image for prediction from: /root/.cache/kagglehub/datasets/nirmalsankalana/plantdoc-dataset/versions/7/train/Apple_Scab_Leaf/train_Apple Scab Leaf_47.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 11s 11s/step
Predicted class: Apple_leaf
